# serialize

> The .ipynb file boundary: serialize the in-memory `nb` (see notebook.py) to a real Jupyter notebook and load one back. A Prompt cell and its Assistant reply are stored solveit-style as a single markdown cell -- the prompt, a `##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_... -->` line, then the reply -- so their structure lives in the cell *source* (which `nbdev-clean` leaves untouched) rather than in metadata (which it strips). Kept apart from the data model (notebook.py) and the rendering/routes (cells.py) so file-format concerns live in one place.

In [ ]:
#| default_exp serialize

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import re, secrets
from pathlib import Path
import nbformat as _nbf
from boopiter.notebook import nb, Notebook, CTYPES
from boopiter.kernel import _MIME_PRIORITY

## The `#| export` pragma

boopiter keeps nbdev's `#| export` pragma out of a cell's editable source entirely, tracking it as a flag instead. `_has_export`/`_strip_export` detect and remove that leading line when reading a cell from disk.

In [ ]:
#| export
# nbdev's '#| export' pragma is a leading line in a code cell's on-disk source. We keep it OUT
# of c.source (and the editor) entirely, tracking it instead as Cell.export (a plain bool) --
# these two helpers are only needed at the load_notebook()/save_notebook() file boundary.
def _has_export(source:str) -> bool:
    "True if `source`'s first line is (some spacing variant of) the '#| export' pragma."
    return source.split('\n', 1)[0].strip().replace(' ', '') == '#|export'

In [ ]:
#| export
def _strip_export(source:str) -> str:
    'The source with any leading #| export pragma line removed.'
    if not _has_export(source): return source
    rest = source.split('\n', 1)
    return rest[1] if len(rest) > 1 else ''

## Saving

`save_notebook` writes `nb` out as a real Jupyter `.ipynb`, merging each Prompt + following Assistant into one markdown cell (solveit separator + the assistant's `<details>` block + the reply). `_blocks_to_nb_outputs` converts boopiter's output blocks into valid nbformat outputs so saved files still render in Jupyter. Cell ids and the separator's reply-id are reused across saves to keep git diffs clean.

In [ ]:
#| export
# boopiter stores a Prompt cell and its Assistant reply as ONE markdown cell on disk, using
# solveit's exact separator -- so the files interoperate with solveit, and (the real point) the
# prompt/reply structure lives in the cell SOURCE, which nbdev-clean leaves untouched, instead of
# in cell metadata, which nbdev-clean strips. On-disk shape of a prompt+reply markdown cell:
#     <prompt source>
#
#     ##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_<hex> -->
#
#     <assistant details block, if any>
#     <reply source>
# The <hex> is the reply's stable id (reused across saves so re-saving doesn't churn git diffs).
_SEP_RE = re.compile(r'\n*##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_([0-9a-f]+) -->\n*')

def _reply_sep(hexid:str) -> str:
    "The solveit reply-separator line, carrying `hexid` as the reply's stable id."
    return f'##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_{hexid} -->'

def _split_reply(reply_blob:str):
    "Split a leading <details>...</details> block (the assistant's model/token info) off the reply text; returns (details_or_None, reply_text)."
    s = reply_blob.lstrip('\n')
    if s.startswith('<details'):
        end = s.find('</details>')
        if end != -1:
            end += len('</details>')
            return s[:end], s[end:].lstrip('\n')
    return None, reply_blob


In [ ]:
#| export
def _blocks_to_nb_outputs(blocks:list[dict]) -> list:
    "Cell.output's block list -> real nbformat outputs, so saved .ipynb files stay valid (and render in GitHub/real Jupyter too)."
    outs = []
    for b in blocks:
        if b['type'] == 'stream':
            outs.append(_nbf.v4.new_output('stream', name='stdout', text=b['data']))
        elif b['type'] == 'error':
            ename, _, evalue = b['data'].partition(': ')
            outs.append(_nbf.v4.new_output('error', ename=ename, evalue=evalue, traceback=[b['data']]))
        else:  # display
            outs.append(_nbf.v4.new_output('display_data', data={b['mime']: b['data']}))
    return outs

def _nb_outputs_to_blocks(outputs:list) -> list[dict]:
    "Inverse of _blocks_to_nb_outputs()."
    blocks = []
    for o in outputs:
        ot = o.get('output_type')
        if ot == 'stream':
            blocks.append({'type':'stream', 'mime':None, 'data':o.get('text', '')})
        elif ot == 'error':
            data = f"{o.get('ename','')}: {o.get('evalue','')}" if o.get('ename') else o.get('evalue', '')
            blocks.append({'type':'error', 'mime':None, 'data':data})
        elif ot in ('display_data', 'execute_result'):
            data = o.get('data', {})
            mime = next((m for m in _MIME_PRIORITY if m in data), next(iter(data), None))
            if mime: blocks.append({'type':'display', 'mime':mime, 'data':data[mime]})
    return blocks

In [ ]:
#| export
def save_notebook(path:str|Path|None=None) -> Path:
    "Serialize `nb.cells` to a real Jupyter notebook file (`{nb.name}.ipynb` in the cwd by default). A Prompt cell and the Assistant reply that follows it are stored solveit-style as a SINGLE markdown cell -- prompt, a `##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_... -->` line, the assistant's `<details>` block, then the reply -- so the prompt/reply structure lives in the cell source (which nbdev-clean leaves alone) rather than in strippable metadata. note/code/raw stay plain native cells; code cells keep their `#| export` pragma and outputs. Cell ids (and the separator's reply-id hex) are preserved across saves so unchanged cells don't churn git diffs. A `solveit_ai:true` flag is also stamped on prompt/reply cells for solveit's benefit -- boopiter itself relies only on the separator."
    path = Path(path) if path else Path.cwd()/f'{nb.name}.ipynb'
    doc = _nbf.v4.new_notebook()
    cells = nb.cells
    i = 0
    while i < len(cells):
        c = cells[i]
        idkw = {'id': c.nb_id} if c.nb_id else {}
        if c.ctype == 'prompt':
            reply = cells[i+1] if i+1 < len(cells) and cells[i+1].ctype == 'assistant' else None
            hexid = reply.nb_id if (reply and reply.nb_id and re.fullmatch(r'[0-9a-f]+', reply.nb_id)) else secrets.token_hex(4)
            parts = [c.source, '', _reply_sep(hexid)]
            if reply is not None:
                if reply.details: parts += ['', reply.details]
                parts += ['', reply.source]
            cell = _nbf.v4.new_markdown_cell('\n'.join(parts), metadata={'solveit_ai': True}, **idkw)
            c.nb_id = cell['id']
            if reply is not None:
                reply.nb_id = hexid   # keep the reply's id stable across saves
                i += 1                # this assistant is folded into the cell above
        elif c.ctype == 'code':
            src = f'#| export\n{c.source}' if c.export else c.source
            outputs = _blocks_to_nb_outputs(c.output) if c.output else []
            cell = _nbf.v4.new_code_cell(src, outputs=outputs, **idkw); c.nb_id = cell['id']
        elif c.ctype == 'raw':
            cell = _nbf.v4.new_raw_cell(c.source, **idkw); c.nb_id = cell['id']
        else:  # note, or a lone assistant with no preceding prompt -> plain markdown
            cell = _nbf.v4.new_markdown_cell(c.source, **idkw); c.nb_id = cell['id']
        doc.cells.append(cell)
        i += 1
    _nbf.write(doc, str(path))
    return path


## Loading

`load_notebook` is the inverse: a markdown cell carrying the `SOLVEIT_SEPARATOR` splits back into a Prompt + Assistant pair (re-extracting the `<details>` block); plain markdown is a note; code cells get their `#| export` flag detected. It falls back to legacy `metadata.boopiter` for notebooks saved by older boopiter, and to nbformat cell types for non-boopiter notebooks.

In [ ]:
#| export
_NB_FALLBACK = {'code':'code', 'markdown':'note', 'raw':'raw'}  # nbformat cell_type -> our ctype, for plain (non-boopiter) notebooks

In [ ]:
#| export
def load_notebook(path:str|Path) -> Notebook:
    "Load a Jupyter notebook file into `nb`, replacing its current contents -- the inverse of save_notebook(). A markdown cell containing the `SOLVEIT_SEPARATOR` splits back into a Prompt cell plus its Assistant reply (with the `<details>` block re-extracted into `details`); other markdown is a note; code cells get their leading `#| export` detected and stripped. For notebooks saved by older boopiter it falls back to reading legacy `metadata.boopiter` (ctype/visible/details); for plain non-boopiter notebooks it falls back to nbformat cell types."
    path = Path(path)
    doc = _nbf.read(str(path), as_version=4)
    nb.cells.clear(); nb._nid = 0; nb.selected = None
    for cell in doc.cells:
        t, src = cell.cell_type, cell.source
        legacy = cell.get('metadata', {}).get('boopiter', {})
        if t == 'code':
            exported = _has_export(src)
            if exported: src = _strip_export(src)
            output = _nb_outputs_to_blocks(cell.get('outputs', [])) or None
            nb.add('code', src, output=output, nb_id=cell.get('id'), export=exported,
                   visible=legacy.get('visible', True))
        elif t == 'raw':
            nb.add('raw', src, nb_id=cell.get('id'), visible=legacy.get('visible', True))
        else:  # markdown: prompt(+assistant), legacy-tagged, or plain note
            m = _SEP_RE.search(src)
            if m:
                nb.add('prompt', src[:m.start()].rstrip('\n'), nb_id=cell.get('id'))
                reply_blob = src[m.end():]
                if reply_blob.strip():
                    details, reply_text = _split_reply(reply_blob)
                    nb.add('assistant', reply_text, details=details, nb_id=m.group(1))
            elif legacy.get('ctype') in CTYPES + ('assistant',):
                nb.add(legacy['ctype'], src, nb_id=cell.get('id'),
                       visible=legacy.get('visible', True), details=legacy.get('details'))
            else:
                nb.add('note', src, nb_id=cell.get('id'))
    nb.name = str(path.with_suffix(''))  # keep the directory, only strip .ipynb
    return nb


## Round-trip test

Proves the point of this whole scheme: save a notebook, wipe every cell's metadata (what `nbdev-clean` does), reload, and confirm the prompt/reply structure -- plus code/note/raw, `#| export`, and the assistant `details` -- all survive because they live in the cell source, not metadata. Runs under `nbdev-test`.

In [ ]:
# Round-trip test: the prompt/reply structure (and code/note/raw, export, details) must survive
# nbdev-clean, which strips cell metadata. We prove it by saving, deleting ALL metadata (exactly
# what nbdev-clean does to the parts we no longer rely on), then reloading and checking.
import tempfile as _tf, json as _json, os as _os
nb.cells.clear(); nb._nid = 0; nb.selected = None
nb.add('note', '# hi there')
nb.add('prompt', 'What is 2+2?')
nb.add('assistant', 'It is 4.', details='<details><summary>Reply details</summary><ul><li>Model: test</li></ul></details>')
nb.add('code', 'x = 4', export=True)
nb.add('raw', 'literal text')
_p = _tf.mktemp(suffix='.ipynb')
save_notebook(_p)
_doc = _json.load(open(_p))                       # simulate nbdev-clean: wipe all metadata
_doc['metadata'] = {}
for _c in _doc['cells']: _c['metadata'] = {}
_json.dump(_doc, open(_p, 'w'))
load_notebook(_p)
assert [c.ctype for c in nb.cells] == ['note','prompt','assistant','code','raw'], [c.ctype for c in nb.cells]
_a = nb.cells[2]
assert _a.ctype == 'assistant' and _a.details and 'Model: test' in _a.details
assert nb.cells[3].export and nb.cells[3].source == 'x = 4'
assert nb.cells[1].source == 'What is 2+2?' and nb.cells[1].ctype == 'prompt'
_os.remove(_p)
print('round-trip survived a full metadata strip:', [c.ctype for c in nb.cells])


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()